In [1]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task2')
RUN_TAG = 'mar23'

/Users/danherma/Documents/projects-personal/scoring_simulator/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
it = pl.read_csv('task2/data/IT.csv')
oot = pl.read_csv('task2/data/OOT.csv')

labeled = it.filter(pl.col('TARGET').is_not_null())
print(f'Labeled: {labeled.shape[0]}, Target rate: {labeled["TARGET"].mean():.4f}')

var_cols = [c for c in it.columns if c.startswith('VARIABLE_')]
cat_cols = [c for c in var_cols if it[c].n_unique() <= 100]
num_cols = [c for c in var_cols if c not in cat_cols]
print(f'Numerical: {len(num_cols)}, Categorical: {len(cat_cols)}')

Labeled: 100001, Target rate: 0.1778
Numerical: 46, Categorical: 13


In [3]:
cutoff = labeled.sort('TIME')['TIME'].quantile(0.8)
train = labeled.filter(pl.col('TIME') <= cutoff)
val = labeled.filter(pl.col('TIME') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')
print(f'Train target: {train["TARGET"].mean():.4f}, Val target: {val["TARGET"].mean():.4f}')

Train: 80001, Val: 20000
Train target: 0.1777, Val target: 0.1780


In [4]:
feat_cols = var_cols
X_train = train.select(feat_cols).to_pandas()
y_train = train['TARGET'].to_numpy().astype(int)
X_val = val.select(feat_cols).to_pandas()
y_val = val['TARGET'].to_numpy().astype(int)
X_oot = oot.select(feat_cols).to_pandas()

In [5]:
bp = BinningProcess(
    variable_names=feat_cols,
    categorical_variables=cat_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

summary = bp.summary().sort_values('iv', ascending=False)
print(summary[['name', 'iv', 'dtype']].head(20).to_string())

Selected (IV>=0.02): 59
             name        iv        dtype
57  VARIABLE_4668  0.244758  categorical
35   VARIABLE_164  0.243432    numerical
15  VARIABLE_4848  0.242115  categorical
30  VARIABLE_5253  0.241679    numerical
43  VARIABLE_3304  0.241637    numerical
13  VARIABLE_4977  0.241354    numerical
44  VARIABLE_3508  0.240263    numerical
18  VARIABLE_3126  0.239683  categorical
5   VARIABLE_4681  0.239349    numerical
54  VARIABLE_3719     0.238  categorical
29  VARIABLE_5017  0.237367    numerical
31  VARIABLE_3862  0.189539  categorical
24  VARIABLE_4847  0.189415    numerical
55   VARIABLE_320  0.188427  categorical
10  VARIABLE_4422   0.18575    numerical
36  VARIABLE_3939  0.185454    numerical
51  VARIABLE_3439  0.183641  categorical
46  VARIABLE_3010  0.183572    numerical
12   VARIABLE_445  0.182149  categorical
9   VARIABLE_3173  0.182097    numerical


In [6]:
X_tr_woe = bp.transform(X_train.values, metric='woe')
X_va_woe = bp.transform(X_val.values, metric='woe')
X_oo_woe = bp.transform(X_oot.values, metric='woe')

best_gini_lr = 0
best_lr = None
best_C = None

for C in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe, y_train)
    p = lr.predict_proba(X_va_woe)[:, 1]
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'WOE+LR C={C}: val_gini={g:.4f}')
    if g > best_gini_lr:
        best_gini_lr, best_lr, best_C = g, lr, C

p_lr_val = best_lr.predict_proba(X_va_woe)[:, 1]
val_gini_lr = best_gini_lr
print(f'\nBest LR: C={best_C}, val_gini={val_gini_lr:.4f}')

WOE+LR C=0.001: val_gini=0.6139
WOE+LR C=0.01: val_gini=0.6311
WOE+LR C=0.05: val_gini=0.6319
WOE+LR C=0.1: val_gini=0.6319
WOE+LR C=0.5: val_gini=0.6319


WOE+LR C=1.0: val_gini=0.6319
WOE+LR C=5.0: val_gini=0.6319

Best LR: C=0.1, val_gini=0.6319


In [7]:
configs = [
    {'num_leaves': 5, 'learning_rate': 0.01, 'min_child_samples': 500, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 3, 'learning_rate': 0.01, 'min_child_samples': 500, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 5, 'learning_rate': 0.005, 'min_child_samples': 500, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 3, 'learning_rate': 0.05, 'min_child_samples': 500, 'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 3.0, 'reg_lambda': 3.0},
    {'num_leaves': 5, 'learning_rate': 0.01, 'min_child_samples': 500, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 3.0, 'reg_lambda': 3.0},
    {'num_leaves': 5, 'learning_rate': 0.01, 'min_child_samples': 300, 'feature_fraction': 0.5, 'bagging_fraction': 0.7, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
]

best_lgb_gini = 0
model = None

for i, cfg in enumerate(configs):
    params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'feature_pre_filter': False, **cfg}
    lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False)
    lgb_va = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)
    m = lgb.train(params, lgb_tr, num_boost_round=8000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
    p = m.predict(X_val)
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'Config {i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]}, iters={m.best_iteration})')
    if g > best_lgb_gini:
        best_lgb_gini, model = g, m

p_lgb_val = model.predict(X_val)
val_gini_lgb = best_lgb_gini
print(f'\nBest LGB: val_gini={val_gini_lgb:.4f}')

Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[2745]	valid_0's auc: 0.816998
Config 0: val_gini=0.6340 (leaves=5, lr=0.01, iters=2745)


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[4820]	valid_0's auc: 0.817187
Config 1: val_gini=0.6344 (leaves=3, lr=0.01, iters=4820)


Training until validation scores don't improve for 200 rounds


...

Config 2: val_gini=0.6341 (leaves=5, lr=0.005, iters=5750)
Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[959]	valid_0's auc: 0.817263
Config 3: val_gini=0.6345 (leaves=3, lr=0.05, iters=959)
Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[2746]	valid_0's auc: 0.817023
Config 4: val_gini=0.6340 (leaves=5, lr=0.01, iters=2746)


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[2570]	valid_0's auc: 0.817051
Config 5: val_gini=0.6341 (leaves=5, lr=0.01, iters=2570)

Best LGB: val_gini=0.6345


In [8]:
best_ens_g = 0
best_w = 0

for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': val_gini_lr, 'LGB_raw': val_gini_lgb, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best overall: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

Best ensemble: w_lr=0.35, val_gini=0.6353
Best overall: Ensemble = 0.6353
  Ensemble: 0.6353
  LGB_raw: 0.6345
  WOE_LR: 0.6319


In [9]:
if best_name == 'WOE_LR':
    p_oot_final = best_lr.predict_proba(X_oo_woe)[:, 1]
elif best_name == 'LGB_raw':
    p_oot_final = model.predict(X_oot)
else:
    p_oot_final = best_w * best_lr.predict_proba(X_oo_woe)[:, 1] + (1 - best_w) * model.predict(X_oot)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task2/predictions.csv')
print(f'OOT predictions saved: {preds.shape}')

with mlflow.start_run(run_name='v6_ultra_simple_lgb'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features', len(feat_cols))
    mlflow.log_param('n_selected_woe', len(selected))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.log_metric('val_gini_lr', val_gini_lr)
    mlflow.log_metric('val_gini_lgb', val_gini_lgb)
    mlflow.log_metric('val_gini_ens', best_ens_g)
    mlflow.set_tag('task', 'task2')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')

OOT predictions saved: (49999, 2)
val_gini: 0.6353
